In [62]:
import numpy as np


X_total = np.load('/home/kbj/dev_ws/unmanned_store/src/X_pose.npy')
y_total = np.load('/home/kbj/dev_ws/unmanned_store/src/y_pose.npy')


In [63]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_total, y_total, test_size=0.2, random_state=42, stratify=y_total)


In [64]:
import torch
from torch.utils.data import Dataset, DataLoader

class PoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = PoseDataset(X_train, y_train)
test_dataset = PoseDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [65]:
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=36, hidden_size=64, num_layers=2, batch_first=True)
        self.fc = nn.Linear(64, 2)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # 마지막 시퀀스만 사용
        out = self.fc(out)
        return out

model = LSTMModel()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


LSTMModel(
  (lstm): LSTM(36, 64, num_layers=2, batch_first=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)

In [66]:
import torch.optim as optim
import torch.nn as nn

# 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

EPOCHS = 50

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(output, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

    acc = correct / total * 100
    print(f"[Epoch {epoch+1:2d}] Loss: {total_loss:.4f} | Train Acc: {acc:.2f}%")

# 🔍 테스트 정확도 평가 함수
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            output = model(X_batch)
            _, predicted = torch.max(output, 1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)
    print(f"📊 Test Accuracy: {correct / total * 100:.2f}%")

# 테스트 정확도 출력
evaluate(model, test_loader)


[Epoch  1] Loss: 4.8740 | Train Acc: 50.00%
[Epoch  2] Loss: 4.8548 | Train Acc: 50.91%
[Epoch  3] Loss: 4.8492 | Train Acc: 53.64%
[Epoch  4] Loss: 4.8527 | Train Acc: 53.18%
[Epoch  5] Loss: 4.8428 | Train Acc: 52.73%
[Epoch  6] Loss: 4.8390 | Train Acc: 51.82%
[Epoch  7] Loss: 4.8347 | Train Acc: 52.73%
[Epoch  8] Loss: 4.8286 | Train Acc: 51.82%
[Epoch  9] Loss: 4.8259 | Train Acc: 54.55%
[Epoch 10] Loss: 4.8229 | Train Acc: 53.18%
[Epoch 11] Loss: 4.8087 | Train Acc: 55.00%
[Epoch 12] Loss: 4.7993 | Train Acc: 57.27%
[Epoch 13] Loss: 4.7940 | Train Acc: 56.36%
[Epoch 14] Loss: 4.7838 | Train Acc: 54.55%
[Epoch 15] Loss: 4.7671 | Train Acc: 56.36%
[Epoch 16] Loss: 4.7520 | Train Acc: 54.55%
[Epoch 17] Loss: 4.7550 | Train Acc: 52.73%
[Epoch 18] Loss: 4.7830 | Train Acc: 51.82%
[Epoch 19] Loss: 4.7311 | Train Acc: 51.36%
[Epoch 20] Loss: 4.6980 | Train Acc: 52.73%
[Epoch 21] Loss: 4.6677 | Train Acc: 58.18%
[Epoch 22] Loss: 4.5672 | Train Acc: 55.91%
[Epoch 23] Loss: 4.5302 | Train 

In [82]:
from sklearn.metrics import confusion_matrix, classification_report

# 예측 수집
y_true, y_pred = [], []
model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        output = model(X_batch)
        pred = output.argmax(dim=1).cpu().numpy()
        y_true.extend(y_batch.cpu().numpy())
        y_pred.extend(pred)

# 출력
print("🔍 Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("📊 Classification Report:\n", classification_report(y_true, y_pred))


🔍 Confusion Matrix:
 [[25  3]
 [ 4 24]]
📊 Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.89      0.88        28
           1       0.89      0.86      0.87        28

    accuracy                           0.88        56
   macro avg       0.88      0.88      0.87        56
weighted avg       0.88      0.88      0.87        56



In [67]:
torch.save(model.state_dict(), "lstm_pose_model.pt")


In [68]:
import cv2
import torch
import numpy as np
from ultralytics import YOLO
import mediapipe as mp


In [75]:
# 모델 불러오기
model = LSTMModel()  # ← 기존에 정의한 LSTM 모델 클래스명으로 교체
model.load_state_dict(torch.load("lstm_pose_model.pt"))
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

# YOLO 및 MediaPipe 설정
yolo_model = YOLO("yolov8s.pt")
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.3)


I0000 00:00:1749104842.837424  617474 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1749104842.839896  737832 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.2.8-1ubuntu1~24.04.1), renderer: Mesa Intel(R) UHD Graphics (CML GT2)


In [76]:
def draw_pose_landmarks(img, landmarks, draw_line, x_offset=0, y_offset=0, crop_width=None, crop_height=None):
    attention_dot = list(range(11, 29))  # 어깨~발만 사용
    draw_dic = {}

    for idx in attention_dot:
        lm = landmarks[idx]
        cx = int(lm.x * crop_width) + x_offset
        cy = int(lm.y * crop_height) + y_offset
        draw_dic[idx] = (cx, cy)
        cv2.circle(img, (cx, cy), 3, (0, 255, 0), -1)

    for p1, p2 in draw_line:
        if p1 in draw_dic and p2 in draw_dic:
            cv2.line(img, draw_dic[p1], draw_dic[p2], (255, 0, 0), 2)

    return img


In [86]:
import torch
import cv2
import numpy as np
from collections import deque

video_path = "/home/kbj/dev_ws/unmanned_store/data/절도_mp4/C_3_12_1_BU_SMA_08-28_13-51-00_CB_RGB_DF2_M1.mp4"
cap = cv2.VideoCapture(video_path)

pose_queue = deque(maxlen=30)  # 최근 30개 pose 시퀀스를 저장
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.resize(frame, (640, 640))

    results = yolo_model.predict(frame, classes=[0], verbose=False)
    boxes = results[0].boxes

    if boxes and len(boxes) > 0:
        confs = boxes.conf.cpu().numpy()
        max_idx = np.argmax(confs)

        x1 = int(boxes.xyxy[max_idx][0]) - 10
        y1 = int(boxes.xyxy[max_idx][1])
        x2 = int(boxes.xyxy[max_idx][2]) + 10
        y2 = int(boxes.xyxy[max_idx][3])
        conf = confs[max_idx]

        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(639, x2)
        y2 = min(639, y2)

        if conf >= 0.3:
            cropped = frame[y1:y2, x1:x2]
            results_pose = pose.process(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))

            if results_pose.pose_landmarks:
                landmarks = results_pose.pose_landmarks.landmark
                attention_dot = [i for i in range(11, 29)]
                pose_data = []
                for i in attention_dot:
                    pose_data.extend([landmarks[i].x, landmarks[i].y])

                # 누적 시퀀스에 추가
                if len(pose_data) == len(attention_dot) * 2:
                    pose_queue.append(pose_data)

                    # 시퀀스가 30개 쌓였으면 예측
                    if len(pose_queue) == 30:
                        input_tensor = torch.tensor([list(pose_queue)], dtype=torch.float32).to(device)
                        output = model(input_tensor)
                        pred = output.argmax(dim=1).item()

                        label = "Normal" if pred == 0 else "Abnormal"
                        color = (0, 255, 0) if label == "Normal" else (0, 0, 255)

                        cv2.putText(frame, f"Prediction: {label}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX,
                                    1, color, 2)

                # 관절 시각화
                draw_line = [[11,13],[13,15],[12,14],[14,16],[11,12],[11,23],[12,24],
                             [23,24],[23,25],[24,26],[25,27],[26,28]]
                frame = draw_pose_landmarks(
                    frame, landmarks, draw_line,
                    x_offset=x1, y_offset=y1,
                    crop_width=(x2 - x1), crop_height=(y2 - y1)
                )

    cv2.imshow("Pose Inference", frame)
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

    frame_idx += 1

cap.release()
cv2.destroyAllWindows()
